<h2>Food (v0.03) - 'Food (v0.03).ipynb'</h2>

<u><h5>Context</h5></u>

The notebook aims to isolate NZF impacts modelled for individual commodities with 'CPC' codes forwarded over by David Forengie on 23rd September 2026.

The objectives of the notebook are:

<ol>
    <li>
        To develop the representation of CPC codes provided by David, as its not completely clear how these work (e.g. where some CPC codes are only 3 digits, whereas others are 5). 
    </li>
    <li>
        To read-in 'v0.64' of the 'Impact' analysis results for the country in question, then isolate these records for the CPC codes identified in the step above.
    </li>
</ol>

<u><h5>Version Documentation</h5></u>

'v0.01' outlined an initial method for completing the process. 'v0.02' refined the method and developed a fully functioning notebook, utilising results from the 'Impact' notebooks which now include CPC codes as fields in the output data tables. Upon inspection, it was found that the identification of CPC codes wasn't functioning properly, where CPC codes specified by David wouldn't show in the dataset, yet HSa4 codes given in the CPC manual would show. This likely explains much of the losses observed when pulling by CPC codes, so 'v0.03' utilises a new version of the codes sent through David which have been mapped to the HS system ahead of time. 

In [1]:
# Import packages
import os
import base64
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.style
plt.style.use('seaborn-v0_8')

def download_link(df, title="Download CSV file", filename="data.csv"):
    csv = df.to_csv()
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

# define function for counting values in each data field
trade_im_field_count = lambda field: trade_im[field].value_counts().to_frame().reset_index()
trade_ex_field_count = lambda field: trade_ex[field].value_counts().to_frame().reset_index()

version = "v0.03"
input_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/"
output_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/food/"

# 1. Set up the notebook

## 1.1 Let's set the country we're interested in and set up a results folder.

In [3]:
# Select one of our countries to work on...
iso_code = "258"
iso_country = "French Polynesia"
year = 2025
import_portfolio = "h5_ag6_by_sea_M.csv" # Imports, '_M' | Foreign Imports, '_FM' | Re-Imports, '_RM'
export_portfolio = "h5_ag6_by_sea_X.csv" # Exports, '_X' | Domestic Exports, '_DX' | Re-exports, '_RX'
gdp = 6.320*1e9

# Create output Journey directory folder if necessary
os.makedirs(output_dir+"{0}/".format(iso_country)) if not os.path.exists(output_dir+"{0}/".format(iso_country)) else "Output directory already exists."

## 1.2 Import and Format 'Impact Analysis' Results

In [4]:
impact_im_c = [
    "trade_idx", "reporterCode", "reporterDesc", "partnerCode", "partnerDesc", "flowDesc", "motDesc", 
    "HS6", "HS6_desc", "HS4", "HS4_desc", "HS2", "HS2_desc", "cpc", "cpc_desc", "eora_desc", "gtap", "gtap_desc", 
    "cifvalue", "fobvalue", "primaryValue", "qtyUnitAbbr", "qty", "netWgt", "vol_kg_vtw", 
    "sea_dist_km", "sea_dist_nm", "tnm", "fac_vessel_types", "fac_vessel_types_str", 
    #"TandT_match_desc", "TandT_ad_val_pct", "TandT_N", "TandT_ciffob", "TandT_cifvalue", "TandT_fobvalue", "cost_int_match_desc",
    "bau_usd_26", "levy_usd_26", "asis_usd_26", "brazil_usd_26", "liscr_usd_26", "direct_usd_26", 
    "bau_usd_29", "levy_usd_29", "asis_usd_29", "brazil_usd_29", "liscr_usd_29", "direct_usd_29", 
    "bau_usd_32", "levy_usd_32", "asis_usd_32", "brazil_usd_32", "liscr_usd_32", "direct_usd_32", 
    "bau_usd_35", "levy_usd_35", "asis_usd_35", "brazil_usd_35", "liscr_usd_35", "direct_usd_35", 
    "bau_usd_38", "levy_usd_38", "asis_usd_38", "brazil_usd_38", "liscr_usd_38", "direct_usd_38", 
    "bau_usd_41", "levy_usd_41", "asis_usd_41", "brazil_usd_41", "liscr_usd_41", "direct_usd_41", 
    "bau_usd_44", "levy_usd_44", "asis_usd_44", "brazil_usd_44", "liscr_usd_44", "direct_usd_44", 
    "bau_usd_47", "levy_usd_47", "asis_usd_47", "brazil_usd_47", "liscr_usd_47", "direct_usd_47", 
    "bau_usd_50", "levy_usd_50", "asis_usd_50", "brazil_usd_50", "liscr_usd_50", "direct_usd_50"	
]

impact_im_d = {
    "reporterCode":"str", "partnerCode":"str", 
    "HS6":"str", "HS4":"str", "HS2":"str", "HS2":"str", "gtap":"str", "cpc":"str"
}

impact_im = pd.read_csv(
    input_dir + "impact/{0}/impact_im_v0.64.csv".format(iso_country), 
    dtype=impact_im_d,
    usecols=impact_im_c
)[impact_im_c]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(impact_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(impact_im.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(impact_im.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(impact_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(impact_im.vol_kg_vtw.sum() / 1e6, 1)))

impact_im["cpc_cl"] = [str(i).split('.')[0] for i in impact_im.cpc.values]
impact_im = impact_im.drop(columns="trade_idx").reset_index(drop=True).reset_index().rename(columns={"index":"trade_idx"})
display(impact_im.head(2))

Aggregate trade portfolio statistics for 'French Polynesia': 

Number of trade records present in dataset: 3024
Number of partner countries present in dataset: 94
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$441.5m
Total weight within the dataset: 548.6 million kilograms


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl
0,0,258,French Polynesia,360,Indonesia,Import,Sea,611693,"611693 | Gloves, mittens and",6116,"6116 | Gloves, mittens and",61,61 | Apparel and clothing,28229,"Gloves, shawls, scarves, veils, ties, cravats ...",Textiles and Wearing Apparel,28,WAP - Wearing apparel,3460.481,NaN,3460.481,kg,124.178,124.178,280.298961,10742.0,6674.767282,1870.930334,['Container'],['Container'],6.705807,6.684415,6.686140,6.686140,6.698880,6.698880,7.041309,13.073465,7.728702,7.185725,7.211683,7.402274,6.882318,12.523461,8.107460,7.463150,7.286681,7.649591,6.742954,13.869252,10.155889,8.860657,8.182702,7.990925,6.605741,14.699013,11.876785,11.048529,8.708254,8.589829,6.471080,15.042854,14.293586,14.533026,9.376177,9.033744,6.333800,14.747747,14.060288,15.024259,11.397834,9.205168,6.204117,13.029951,13.509996,14.147409,10.413657,9.316696,6.072633,11.806180,12.040855,13.980849,11.600967,9.388171,28229
1,1,258,French Polynesia,360,Indonesia,Import,Sea,630210,630210 | Bed linen; knitted o,6302,"6302 | Bed linen, table lin",63,"63 | Textiles, made up ar",2712,"Bed linen, table linen, toilet linen and kitch...",Textiles and Wearing Apparel,27,TEX - Textiles,6388.640,NaN,6388.640,kg,933.845,933.845,797.302272,10742.0,6674.767282,5321.807119,['Container'],['Container'],19.074475,19.013625,19.018533,19.018533,19.054769,19.054769,20.028798,37.187093,21.984068,20.439586,20.513423,21.055553,19.576554,35.622622,23.061435,21.228715,20.726754,21.759041,19.180136,39.450685,28.888133,25.203882,23.275458,22.729955,18.789840,41.810916,33.783172,31.427222,24.770377,24.433519,18.406798,42.788962,40.657691,41.338771,26.670265,25.696223,18.016311,41.949539,39.994083,42.736070,32.420809,26.183834,17.647431,37.063319,38.428792,40.241894,29.621347,26.501071,17.273428,33.582336,34.249864,39.768119,32.998614,26.704382,2712


# 2. Subset the 'Impact Analysis' results for food security-related CPC codes

## 2.1 Let's get a sense of how much of import value is associated with food-relevant goods across classification systems

### 2.1.1 Harmonised System (HS01-23)

In [5]:
#print("HS groupings: \n\n{0}\n".format(pd.Series(impact_im.HS2_desc.unique()).sort_values().values))
HSa2_good_groups = [
    "01", "02", "03", "04", "05", "07", "08", # "06", 
    "09", "10", "11", "12", "13", "14", "15", "16", 
    "17", "18", "19", "20", "21", "22", "23"
]

impact_im_HSa2_food = impact_im[(impact_im.HS2.isin(HSa2_good_groups))]
impact_im_HSa2_food_g = impact_im_HSa2_food.groupby(["HS2_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_HSa2_food_g["pct_val"] = round(100 * impact_im_HSa2_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_HSa2_food_g["pct_vol"] = round(100 * impact_im_HSa2_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_HSa2_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_HSa2_food_g.round(1)
))

Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): $196.2m (44.4%), ... 

                             cifvalue   vol_kg_vtw  pct_val  pct_vol
HS2_desc                                                            
02 | Meat and edible meat  65664133.8   27219138.0     14.9      5.0
20 | Preparations of vege  19333376.3   13452268.4      4.4      2.5
16 | Meat, fish, crustace  15092975.8    2948357.5      3.4      0.5
19 | Preparations of cere  13199723.1    5283084.9      3.0      1.0
10 | Cereals               11304959.5   20096688.6      2.6      3.7
21 | Miscellaneous edible  10730794.4    3858623.9      2.4      0.7
07 | Vegetables and certa   9610422.5   13934210.5      2.2      2.5
15 | Animal, vegetable or   8800779.3    7258274.7      2.0      1.3
08 | Fruit and nuts, edib   7030010.2    5685861.9      1.6      1.0
22 | Beverages, spirits a   6369031.1  211077294.3      1.4     38.5
04 | Dairy produce; birds   5895389.7    4436329.4      1.3      0.8
17

### 2.1.2 EORA groupings ('Food & Beverages' and 'Fishing')

In [6]:
print("EORA groupings: \n\n{0}\n".format(pd.Series(impact_im.eora_desc.unique()).sort_values().values))
eora_good_groups = ["Food & Beverages", "Agriculture", "Fishing"]

impact_im_eora_food = impact_im[(impact_im.eora_desc.isin(eora_good_groups))]
impact_im_eora_food_g = impact_im_eora_food.groupby(["eora_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_eora_food_g["pct_val"] = round(100 * impact_im_eora_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_eora_food_g["pct_vol"] = round(100 * impact_im_eora_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_eora_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_eora_food_g.round(1)
))

EORA groupings: 

['Agriculture' 'Electrical and Machinery' 'Fishing' 'Food & Beverages'
 'Metal Products' 'Mining and Quarrying' 'Other Manufacturing'
 'Petroleum, Chemical and Non-Metallic Mineral Products'
 'Textiles and Wearing Apparel' 'Transport Equipment' 'Wood and Paper' nan]

Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): $181.8m (41.2%), ... 

                     cifvalue   vol_kg_vtw  pct_val  pct_vol
eora_desc                                                   
Agriculture       108812674.1   82759618.2     24.6     15.1
Food & Beverages   72686202.0  250851512.5     16.5     45.7
Fishing              325362.6      57278.9      0.1      0.0



### 2.1.3 GTAP (16x headings)

In [7]:
#print("GTAP groupings: \n\n{0}\n".format(pd.Series(impact_im.gtap_desc.unique()).sort_values().values))
gtap_food_groups = [
    "GRO - Cereal grains n.e.c.", "PDR - Paddy rice", "PCR - Processed rice", "SGR - Sugar", 
    "CMT - Bovine meat prods", "CTL - Bovine cattle, sheep and goats, horses", "FSH - Fishing", "OMT - Meat products n.e.c.",
    "MIL - Dairy products", "OFD - Food products n.e.c.", "V_F - Vegetables, fruit, nuts", "OSD - Oil seeds", 
    "OCR - Crops n.e.c.", "PFB - Plant-based fibers", "VOL - Vegetable oils and fats", 
    "OAP - Animal products n.e.c."
]

impact_im_gtap_food = impact_im[(impact_im.gtap_desc.isin(gtap_food_groups))]
impact_im_gtap_food_g = impact_im_gtap_food.groupby(["gtap_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_gtap_food_g["pct_val"] = round(100 * impact_im_gtap_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_gtap_food_g["pct_vol"] = round(100 * impact_im_gtap_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via GTAP: ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_gtap_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_gtap_food_g.head(15).round(1)
))

Value of food imports identified via GTAP: $176.2m (39.9%), ... 

                                 cifvalue  vol_kg_vtw  pct_val  pct_vol
gtap_desc                                                              
OFD - Food products n.e.c.     61738043.8  37100494.4     14.0      6.8
OMT - Meat products n.e.c.     43234457.0  21124151.7      9.8      3.9
CMT - Bovine meat prods        27203012.1   6143457.1      6.2      1.1
V_F - Vegetables, fruit, nuts  11965863.3  15556360.8      2.7      2.8
PCR - Processed rice           10993134.2  19095074.2      2.5      3.5
VOL - Vegetable oils and fats   8729244.7   7192956.5      2.0      1.3
SGR - Sugar                     5210961.8  11341959.2      1.2      2.1
MIL - Dairy products            4511877.8   2064753.8      1.0      0.4
OAP - Animal products n.e.c.    1744184.0   1713399.8      0.4      0.3
OCR - Crops n.e.c.               462567.5    172694.8      0.1      0.0
GRO - Cereal grains n.e.c.       185413.3    766962.3      0.0      0.

## 2.2 Comparison of results

In [8]:
print("Share of total import value associated with food-relevant goods by classification system...\n\nHSa2 {0}% \nGTAP {1}% \nEORA {2}%".format(
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))

Share of total import value associated with food-relevant goods by classification system...

HSa2 44.4% 
GTAP 39.9% 
EORA 41.2%


# 3. Take HSa4 codes associated with specific CPCs

## 3.1 Set HSa4 Codes we're interested in

In [9]:
hsa4_codes_food = {
    "Dairy": [
        "0401", "0402", "0403", "0404", "0405", "0406",
    ],
    "Fats and Oils": [
        "1501", "1502", "1503", "1504", "1505", "1506", "1507", "1508", "1509", "1510"
        "1511", "1512", "1513", "1514", "1515", "1516", "1517"
    ],
    "Fruits": [
        "0709", "0801", "0802", "0803", "0804", "0805", "0806", "0807", "0808", "0809", 
        "0810", "0813", "2009"
    ],
    "Meat and Prep": [
        "0201", "0203", "0204", "0205", "0207", "0208", "0209", "2301"
    ],
    "Processed Cereals": [
        "1006", "1101", "1102", "1104", "1105", "1106", "1107", "1108", "1109", "1302",
        "1602", "1901", "1902", "1903", "1904", "1905", "2005", "2007", "2104", "2106", 
        "3505"
    ],
    "Sugar and Honey": [
        "1701", "1702", "1703", "1704", "2006"
    ],
    "Unprocessed Cereals": [
        "1001", "1002", "1003", "1004", "1005", "1007", "1008"
    ],
    "Vegetables": [
        "0701", "0702", "0703", "0704", "0705", "0706", "0707", "0708", "0709", "0710",
        "0712", "0713", "0714", "1209", "2009"
    ]
}

## 3.2 Identify 'Food Security' groups of interest via CPC

In [10]:
impact_im["food_group"] = np.nan

for food_group in list(hsa4_codes_food.keys()):
    impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group

display(impact_im.head(2))

/var/folders/_k/81z8ryjd28lc4f0bb4dm7gkw0000gn/T/ipykernel_27820/1412186970.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Dairy' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl,food_group
0,0,258,French Polynesia,360,Indonesia,Import,Sea,611693,"611693 | Gloves, mittens and",6116,"6116 | Gloves, mittens and",61,61 | Apparel and clothing,28229,"Gloves, shawls, scarves, veils, ties, cravats ...",Textiles and Wearing Apparel,28,WAP - Wearing apparel,3460.481,NaN,3460.481,kg,124.178,124.178,280.298961,10742.0,6674.767282,1870.930334,['Container'],['Container'],6.705807,6.684415,6.686140,6.686140,6.698880,6.698880,7.041309,13.073465,7.728702,7.185725,7.211683,7.402274,6.882318,12.523461,8.107460,7.463150,7.286681,7.649591,6.742954,13.869252,10.155889,8.860657,8.182702,7.990925,6.605741,14.699013,11.876785,11.048529,8.708254,8.589829,6.471080,15.042854,14.293586,14.533026,9.376177,9.033744,6.333800,14.747747,14.060288,15.024259,11.397834,9.205168,6.204117,13.029951,13.509996,14.147409,10.413657,9.316696,6.072633,11.806180,12.040855,13.980849,11.600967,9.388171,28229,NaN
1,1,258,French Polynesia,360,Indonesia,Import,Sea,630210,630210 | Bed linen; knitted o,6302,"6302 | Bed linen, table lin",63,"63 | Textiles, made up ar",2712,"Bed linen, table linen, toilet linen and kitch...",Textiles and Wearing Apparel,27,TEX - Textiles,6388.640,NaN,6388.640,kg,933.845,933.845,797.302272,10742.0,6674.767282,5321.807119,['Container'],['Container'],19.074475,19.013625,19.018533,19.018533,19.054769,19.054769,20.028798,37.187093,21.984068,20.439586,20.513423,21.055553,19.576554,35.622622,23.061435,21.228715,20.726754,21.759041,19.180136,39.450685,28.888133,25.203882,23.275458,22.729955,18.789840,41.810916,33.783172,31.427222,24.770377,24.433519,18.406798,42.788962,40.657691,41.338771,26.670265,25.696223,18.016311,41.949539,39.994083,42.736070,32.420809,26.183834,17.647431,37.063319,38.428792,40.241894,29.621347,26.501071,17.273428,33.582336,34.249864,39.768119,32.998614,26.704382,2712,NaN


## 3.3 Inspect the output

In [11]:
impact_im_cpc_food = impact_im[(~impact_im.food_group.isna())]
impact_im_cpc_food_g = impact_im_cpc_food.groupby("food_group").agg({"HS6_desc":"count", "cifvalue":"sum", "vol_kg_vtw":"sum"}).reset_index().sort_values(by="cifvalue", ascending=False).round(2).rename(columns={"HS6_desc":"N_com"}).reset_index(drop=True)
impact_im_cpc_food_g["pct_val"] = round(100 * impact_im_cpc_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_cpc_food_g["pct_vol"] = round(100 * impact_im_cpc_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Proportion of imports identified as food security-relevant:\n")
print("Value: ${0}m of ${1}m ({2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / 1e6, 1),
    round(impact_im.cifvalue.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))
print("Volume: {0} of {1} million kilograms ({2}%)".format(
    round(impact_im_cpc_food.vol_kg_vtw.sum() / 1e6, 1),
    round(impact_im.vol_kg_vtw.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.vol_kg_vtw.sum() / impact_im.vol_kg_vtw.sum(), 1)
))
print("Value-to-Weight: {0} from {1} US$ per kg (+{2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum(), 2),
    round(impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum(), 2),
    round(100 * ((impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum()) - (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum())) / (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum()), 1)
))
print("Average distance to market (volume-weighted): {0} from {1} nm (+{2}%)".format(
    round((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000), 1),
    round((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000), 1),
    round(100 * (((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000)) - ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000))) / \
          ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000)), 1)
))
print("\n{0}\n".format(impact_im_cpc_food_g.head(15).round(1)))

impact_im_cpc_food_g.to_csv(output_dir + "{0}/food_total_usd_{1}.csv".format(iso_country, version), index=False)

Proportion of imports identified as food security-relevant:

Value: $119.8m of $441.5m (27.1%)
Volume: 101.8 of 548.6 million kilograms (18.6%)
Value-to-Weight: 1.18 from 0.8 US$ per kg (+46.2%)
Average distance to market (volume-weighted): 6204.2 from 8165.1 nm (+-24.0%)

            food_group  N_com    cifvalue  vol_kg_vtw  pct_val  pct_vol
0        Meat and Prep     19  41511785.9  21541735.1      9.4      3.9
1    Processed Cereals    255  40533174.2  35629929.2      9.2      6.5
2           Vegetables     82  10891383.5  15002403.9      2.5      2.7
3        Fats and Oils     20   8800779.3   7258274.7      2.0      1.3
4               Fruits     57   6608298.5   5492127.1      1.5      1.0
5      Sugar and Honey     39   6026596.9  11750235.0      1.4      2.1
6                Dairy     23   5102490.1   4155405.1      1.2      0.8
7  Unprocessed Cereals      4    286045.4    956834.8      0.1      0.2



## 3.3 Derive results for 2029, 2041 and 2050

### 3.3.1 Total FEM costs by scenario (US$)

In [12]:
print("\nTotal FEM costs by scenario (US$)...\n")

d = {"food_group": list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["bau", "levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            round(100 * impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum(), 1) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$)...



,food_group,bau_29,levy_29,asis_29,brazil_29,liscr_29,direct_29,bau_41,levy_41,asis_41,brazil_41,liscr_41,direct_41,bau_50,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,7720903.1,14276343.7,8464675.3,7874777.2,7902631.3,8112908.0,7100892.7,16421671.2,15607028.9,15879433.0,10261037.4,9886174.6,6667940.6,12913559.2,13166022.8,15277375.2,12674812.5,10275150.7
1,Fats and Oils,27836432.3,51501359.5,30523105.5,28393511.4,28494247.6,29251698.6,25598378.2,59243306.8,56302609.1,57279646.0,37004801.5,35652992.0,24035405.1,46574272.9,47487001.7,55107303.6,45720521.8,37055222.1
2,Fruits,7940699.9,14436174.0,8664703.1,8080203.3,8106242.8,8328330.5,7325015.6,16582271.0,15770239.3,16090336.3,10471863.8,10087681.1,6896155.7,13160310.9,13398156.3,15494252.4,12842572.3,10488325.6
3,Meat and Prep,48743870.6,90501827.2,53502388.4,49743599.0,49923295.2,51242672.9,44796426.2,104135036.4,98948183.6,100605721.9,64907136.2,62536619.9,42038154.5,81728969.2,83353525.7,96783243.0,80308369.7,64990165.6
4,Processed Cereals,96547057.5,178672520.3,105873901.1,98482805.2,98832964.9,101458924.4,88780538.7,205551230.0,195343425.5,198721182.4,128372635.5,123673315.9,83356283.5,161564756.3,164734970.4,191180231.5,158623113.2,128533552.0
5,Sugar and Honey,20650884.6,38161298.0,22638593.8,21060836.8,21136274.1,21698636.2,18994838.9,43958025.5,41769898.5,42489392.3,27479638.9,26434982.7,17837768.6,34537117.5,35213259.0,40865010.2,33928451.2,27463376.3
6,Unprocessed Cereals,1957516.2,3535737.8,2132040.3,1990164.7,1996427.9,2051542.7,1807809.2,4063309.9,3865917.2,3948246.7,2575435.8,2478818.1,1703625.7,3228984.0,3286242.2,3799576.8,3150954.1,2577222.1
7,Vegetables,28964267.9,53123468.6,31681775.9,29507538.7,29608646.6,30407251.2,26678039.0,61116705.6,58094822.4,59180289.7,38369548.8,36950462.8,25079110.1,48204479.4,49131386.3,56953823.0,47235028.8,38402959.6


### 3.3.2 Total FEM costs by scenario (% cif)

In [13]:
print("\nTotal FEM costs by scenario (% cif)...\n")

d = {"food_group":list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            np.round(100 * (impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum() - impact_im[(impact_im.food_group == hsa4_codes_food_key)]["bau_usd_{1}".format(scenario, year)].sum()) / \
                     impact_im[(impact_im.food_group == hsa4_codes_food_key)].cifvalue.sum(), 2) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_pct_cif_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (% cif)...



,food_group,levy_29,asis_29,brazil_29,liscr_29,direct_29,levy_41,asis_41,brazil_41,liscr_41,direct_41,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,1.28,0.15,0.03,0.04,0.08,1.83,1.67,1.72,0.62,0.55,1.22,1.27,1.69,1.18,0.71
1,Fats and Oils,2.69,0.31,0.06,0.07,0.16,3.82,3.49,3.60,1.30,1.14,2.56,2.66,3.53,2.46,1.48
2,Fruits,0.98,0.11,0.02,0.03,0.06,1.40,1.28,1.33,0.48,0.42,0.95,0.98,1.30,0.90,0.54
3,Meat and Prep,1.01,0.11,0.02,0.03,0.06,1.43,1.30,1.34,0.48,0.43,0.96,1.00,1.32,0.92,0.55
4,Processed Cereals,2.03,0.23,0.05,0.06,0.12,2.88,2.63,2.71,0.98,0.86,1.93,2.01,2.66,1.86,1.11
5,Sugar and Honey,2.91,0.33,0.07,0.08,0.17,4.14,3.78,3.90,1.41,1.23,2.77,2.88,3.82,2.67,1.60
6,Unprocessed Cereals,5.52,0.61,0.11,0.14,0.33,7.89,7.20,7.48,2.68,2.35,5.33,5.53,7.33,5.06,3.05
7,Vegetables,2.22,0.25,0.05,0.06,0.13,3.16,2.88,2.98,1.07,0.94,2.12,2.21,2.93,2.03,1.22
